In [2]:
import pyarrow.dataset as ds
import pandas as pd
from pathlib import Path
import numpy as np
import statsmodels.api as sm
import os
from tqdm import tqdm
import scipy

In [3]:
ROOT = Path("/home/hanwenying")
DATA = ROOT / "rothman-data/gltd"
OUT  = ROOT / "rothman-anna/gltd/out"
BETA = OUT / "betas"
RESID = OUT / "resid"
SUMMARY = OUT / "summary"

In [4]:
metadata = pd.read_csv(OUT / "metadata.tsv", sep='\t')

In [5]:
resids = list(RESID.glob('*.tsv'))

In [6]:
summary = pd.DataFrame(columns=['TISSUE', 'RHO_BETA', 'RHO_RESID', 'NGENES', 'NSAMPLES'])

In [7]:
for path in resids:
	tissuedf = pd.read_csv(path, sep='\t')

	rho_betas, _ = scipy.stats.spearmanr(tissuedf['LOGLENGTH'], tissuedf['BETA'])
	rho_resid, _ = scipy.stats.spearmanr(tissuedf['LOGLENGTH'], tissuedf['RESIDUAL'])

	nsamples = len(metadata[metadata['SMTSD'] == path.stem])

	summary.loc[len(summary)] = {'TISSUE': path.stem, 'RHO_BETA': rho_betas, 'RHO_RESID': rho_resid, 'NGENES': len(tissuedf), 'NSAMPLES': nsamples}


In [ ]:
summary.to_csv(OUT / "tissuesummary.tsv", sep='\t', index=False)